# 徒手實作簡化版的 RAG 教學  

## 使用 Groq `llama-3.3-70b-versatile` + `intfloat/multilingual-e5-small`

### 本講義要帶你完成什麼？
這份 Notebook 的目標，是用**最少的程式碼**理解 RAG（Retrieval-Augmented Generation，檢索增強生成）的核心流程。  
你會親手完成一個最小可執行版本，並理解它背後的設計邏輯。

---

## 學習目標
完成本講義後，你應該能回答以下問題：

1. 什麼是 RAG？
2. 為什麼不能只靠大型語言模型直接回答？
3. embedding 是什麼？它在 RAG 中扮演什麼角色？
4. 為什麼這裡選用 `intfloat/multilingual-e5-small`？
5. 為什麼查詢與文件要分別加上 `query:` 與 `passage:` 前綴？
6. 檢索到的內容，是如何被送進 LLM 生成答案的？

---

## 這份 Notebook 的流程
我們會依序完成四件事：

### 第一步：建立小型知識庫
先準備幾筆文字資料，模擬外部知識來源。

### 第二步：將文字轉成向量
使用 `intfloat/multilingual-e5-small` 把每段文字轉成 embedding。

### 第三步：做相似度搜尋
把使用者問題也轉成向量，再和知識庫向量比較，找出最相關的內容。

### 第四步：交給大型語言模型生成答案
把檢索結果當成上下文，交給 Groq 的 `llama-3.3-70b-versatile` 回答。

---

## 本講義使用的模型
### LLM（生成模型）
- `llama-3.3-70b-versatile`

### Embedding 模型
- `intfloat/multilingual-e5-small`

---

## 重要觀念先記住
### 1. embedding 維度要設多少？
`intfloat/multilingual-e5-small` 的輸出維度是 **384**。  
也就是說，未來如果你把向量存到資料庫（例如 pgvector、Milvus、FAISS），向量欄位維度都要設定成 **384**。

### 2. E5 系列模型的前綴規則
這個模型建議這樣使用：

- 查詢文字前面加上：`query: `
- 文件文字前面加上：`passage: `

例如：
- `query: RAG 是什麼？`
- `passage: RAG 是結合檢索和生成技術的模型。`

這樣做的原因是：模型在訓練時，查詢與文件本來就是用不同角色來學習的。  
如果不加前綴，效果通常會變差。

### 3. 這份範例是「教學版」
這裡的 corpus 很小，主要目的是幫助你看懂流程。  
真正的企業應用通常還會加入：
- 文件切塊（chunking）
- 向量資料庫
- 門檻值設定
- Top-k 檢索策略
- 引用來源
- 防幻覺提示詞設計

## 步驟一：建立知識庫並產生向量

### 這一格在做什麼？
這一格同時完成了 RAG 中最核心的兩件事：

1. 載入 embedding 模型  
2. 把知識庫文字轉成向量

---

## 你應該注意的重點

### A. 為什麼用 `SentenceTransformer`？
因為它可以很方便地載入 Hugging Face 上的 embedding 模型，並直接把句子轉成向量。

### B. 為什麼 `EMBEDDING_DIM = 384`？
因為 `intfloat/multilingual-e5-small` 的輸出維度就是 **384**。  
這不是自己隨便填的，而是模型本身決定的。

### C. 為什麼文件要加 `passage: `？
E5 模型在訓練時，會區分「文件」與「查詢」的角色。  
知識庫裡的文本屬於文件，因此要加 `passage: `。

### D. 為什麼這裡要 `normalize_embeddings=True`？
因為我們等一下要做 cosine similarity（餘弦相似度）比較。  
先做正規化，會讓向量比較更穩定，也比較符合這類語意搜尋的常見做法。

---

## 執行後你應該觀察什麼？
請看輸出是否包含：

- 使用的 embedding model 名稱
- embedding 維度是否為 384
- 檢索到的文本是不是和問題「RAG 是什麼？」有關

如果你看到的結果很合理，代表最基本的檢索流程已經成立了。

In [1]:
# 安裝必要套件
# !pip install -q sentence-transformers openai scikit-learn

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 載入 embedding model
embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")

# multilingual-e5-small 的輸出維度
EMBEDDING_DIM = 384
print("Embedding model:", "intfloat/multilingual-e5-small")
print("Embedding dimension:", EMBEDDING_DIM)

# 建立資料庫文本
corpus = [
    "如何實作 RAG？",
    "RAG 是結合檢索和生成技術的模型。",
    "RAG 可以從外部資料庫檢索訊息來增強生成模型的回應。",
    "嵌入和向量資料庫是 RAG 的基礎。"
]

# E5 建議：文件加上 passage: 前綴
passages = [f"passage: {text}" for text in corpus]
corpus_vectors = embedding_model.encode(passages, normalize_embeddings=True)

# 使用者查詢
query = "RAG 是什麼？"

# E5 建議：查詢加上 query: 前綴
query_vector = embedding_model.encode([f"query: {query}"], normalize_embeddings=True)

# 計算相似度
similarities = cosine_similarity(query_vector, corpus_vectors)

# 找出最相似的文本
# argsort() 不會直接回傳排序後的分數，它會回傳「由小到大排序後的索引位置」
# [-2:] 這表示取「最後兩個索引」。因為 argsort() 是從小排到大，最後兩個就是最大的兩個分數。
# [::-1] 這是把順序反過來，變成由大到小。

top_k_indices = similarities[0].argsort()[-2:][::-1]
retrieved_texts = [corpus[idx] for idx in top_k_indices]
print("檢索到的文本:", retrieved_texts)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model: intfloat/multilingual-e5-small
Embedding dimension: 384
檢索到的文本: ['RAG 是結合檢索和生成技術的模型。', '如何實作 RAG？']


## 小觀察：目前檢索到哪些文本？

這一格只是把上一格找到的 `retrieved_texts` 直接顯示出來。  

### 這一步有什麼教學意義？
在做 RAG 時，最重要的不只是「模型有沒有回答」，而是你要先確認：

- 檢索出來的文本對不對
- 檢索品質是否合理
- 問題與文件的語意是否真的被向量捕捉到

很多 RAG 專案失敗，不是 LLM 太弱，而是**檢索階段就拿錯資料**。

所以請養成一個習慣：  
**先檢查 retrieved texts，再檢查最終回答。**

In [2]:
d = retrieved_texts


## 步驟二：把檢索結果交給 LLM 生成答案

### 這一格在做什麼？
這一格把剛剛檢索到的文本串起來，當成「上下文（context）」送進大型語言模型。  
這就是 RAG 的核心精神：

> 不是只靠模型腦中的既有知識，而是先去找資料，再根據資料回答。

---

## 你要注意的幾個地方

### A. 為什麼 API Key 要用 `GROQ_API_KEY`？
因為這份 Notebook 是透過 Groq 的 API 來呼叫模型。  
在 Colab 中，通常會把金鑰放在 `userdata` 裡管理。

### B. 為什麼 `OpenAI` 可以拿來呼叫 Groq？
因為 Groq 提供了相容 OpenAI API 格式的介面，  
所以只要改 `base_url`，就可以沿用相同的呼叫方式。

### C. 為什麼把檢索內容放在 `system` 裡？
這是一種簡化寫法，目的是告訴模型：

- 先參考這段上文
- 再根據上文回答使用者問題

在更正式的專案中，也有人會把 context 放在 user prompt，或設計成更明確的 prompt template。

---

## 執行後請觀察
請比較模型的回答是否明顯參考了剛剛檢索到的內容。  
如果有，代表「檢索 → 生成」這個流程已經串起來了。

In [3]:
# 整合檢索文本並傳入 Groq LLM
d = " ".join(retrieved_texts)

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()  # 從 .env 檔案載入環境變數

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": f"根據以下上文回答使用者問題：{d}"},
        {"role": "user", "content": query}
    ]
)

print("生成的回答:", completion.choices[0].message.content)


生成的回答: RAG（Retrieve, Augment, Generate）是一種結合檢索和生成技術的模型。它旨在通過檢索相關的知識或信息，然後根據這些信息生成答案或內容。

RAG 模型通常由三個部分組成：

1. **檢索（Retrieve）**：此步驟涉及到搜索和選擇相關的知識或信息，以用於生成答案或內容。
2. **增強（Augment）**：在這一步，模型可能會對檢索到的信息進行處理和增強，例如，對信息進行過濾、排序或合併等。
3. **生成（Generate）**：最終，模型根據增強過的信息生成答案或內容。

RAG 模型可以被應用於多種任務中，如答案生成、對話生成、文本摘要等。它可以更好地利用已有的知識和信息，生成更加準確和相關的答案或內容。


## 對照實驗：如果不給上下文，模型會怎麼回答？

這一格非常重要，因為它讓你看見 **RAG 的價值**。

### 這一格在做什麼？
同樣問模型一個問題，但是這次**不提供任何檢索到的文本**。  
也就是說，模型只能依賴它原本參數中的知識來回答。

---

## 你應該比較什麼？
請把這一格的結果和前一格的結果做比較：

- 哪一個回答比較貼近我們的知識庫內容？
- 哪一個比較容易出現泛泛而談？
- 如果今天問題是企業內部文件，沒有 context 時模型還能答對嗎？

---

## 重要理解
LLM 很會「說得像知道」，但不代表它真的依據你的資料回答。  
RAG 的價值就在於：

- 讓回答更貼近外部知識
- 讓答案更可控
- 降低只靠模型記憶造成的偏差與幻覺

這也是企業導入 RAG 的主要原因之一。

In [4]:
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": query}
    ]
)

print("生成的回答:", completion.choices[0].message.content)  # 沒有給定上下文時，模型只能依賴自身知識回答


生成的回答: RAG 可能有多個解釋，以下是一些常見的意思：

1. **紅綠燈**（Red, Amber, Green）：RAG 是紅綠燈的簡稱，常用於交通信號燈、工作安全或項目管理中，用於表示狀態或進度的不同階段。
2. **RAG 報告**（Red, Amber, Green Report）：在項目管理中，RAG 報告是一種簡單的報告方式，使用紅、黃、綠色來表示項目的狀態，紅色代表高風險或嚴重問題，黃色代表需要關注的問題，綠色代表正常或低風險。
3. **RAG 評估**（Red, Amber, Green Assessment）：RAG 評估是一種簡單的評估方法，使用紅、黃、綠色來評估某個項目或情況的風險或狀態。

如果您有更多的上下文或詳情，我可以提供更具體的解釋。


## 檢查向量形狀：理解 embedding 維度

這一格會顯示 `corpus_vectors.shape`。  

### 你應該怎麼讀？
如果你的知識庫有 4 筆資料，而且每筆都被轉成 384 維向量，  
那結果通常會是：

```python
(4, 384)
```

意思是：
- 4：共有 4 段文本
- 384：每段文本都被表示成 384 維向量

---

## 這件事為什麼重要？
因為未來你只要接觸向量資料庫，一定會碰到這個概念。  
例如：
- pgvector 欄位要設 `vector(384)`
- FAISS index 的維度也要是 384
- 如果維度錯了，系統就會報錯

所以「模型維度」不是小細節，而是實作中很重要的基礎設定。

In [5]:
corpus_vectors.shape


(4, 384)

## 看看向量內容長什麼樣子

這一格把向量轉成 DataFrame 顯示，讓你具體感受 embedding 的形式。

### 教學重點
embedding 並不是人類可直接閱讀的語意標籤，  
它是一串數值，用來在高維空間中表示文字語意位置。

你不需要逐一解讀每個維度的意義，  
但你要理解一件事：

> 語意相近的句子，向量位置通常也會比較接近。

這就是為什麼我們可以透過相似度計算，找出與問題最接近的文本。

---

## 常見誤解提醒
很多初學者會問：
「dim_0 是不是代表主題？dim_1 是不是代表情緒？」

通常不是這樣理解。  
embedding 的每一維不是人類直觀定義的欄位，而是模型學出來的分散式表示。

In [6]:
import pandas as pd

pd.DataFrame(corpus_vectors, columns=[f"dim_{i}" for i in range(corpus_vectors.shape[1])]).head()


,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,...,dim_374,dim_375,dim_376,dim_377,dim_378,dim_379,dim_380,dim_381,dim_382,dim_383
0,0.091966,-0.033801,-0.073421,-0.082011,0.090159,-0.055938,0.009816,0.058049,0.068183,-0.015189,...,-0.041183,-0.071861,-0.065728,0.052296,-0.045672,-0.039827,0.042065,0.050382,0.040338,0.070381
1,0.053998,-0.015602,-0.066379,-0.058749,0.079863,-0.013701,0.077855,0.078441,0.090574,-0.018370,...,-0.019543,-0.059310,-0.072694,0.044671,-0.044198,-0.056365,0.054279,0.059460,0.059931,0.082523
2,0.102959,-0.037357,-0.047851,-0.087707,0.086237,-0.009663,0.065819,0.065435,0.074973,-0.026943,...,-0.025326,-0.064845,-0.047417,0.015821,-0.044643,-0.015277,0.035721,0.049083,0.006903,0.075504
3,0.051590,-0.025450,-0.045126,-0.049959,0.079211,-0.033181,0.039698,0.080710,0.089622,-0.018434,...,-0.045619,-0.052135,-0.068483,0.022346,-0.022562,-0.056175,0.072948,0.041772,0.056194,0.084301


## 換一個問題試試看：當知識庫沒有答案時會發生什麼事？

這裡把問題改成：`Agent 是什麼？`

### 你要觀察什麼？
請先看檢索到的文本是否真的和 Agent 有關。  
如果知識庫裡根本沒有 Agent 的內容，RAG 的檢索結果就可能會：

- 找到語意上勉強接近的內容
- 或找到其實不太相關的句子

接著，LLM 仍然可能根據這些不夠準確的內容回答。

---

## 這告訴我們什麼？
RAG 並不是魔法。  
如果知識庫沒有答案，或檢索拿錯資料，最終回答仍然可能失準。

所以真正的實務重點不只是「有沒有接上 LLM」，  
而是整體資訊鏈條是否可靠：

1. 知識庫內容夠不夠完整  
2. chunk 切得好不好  
3. embedding 模型選得對不對  
4. 檢索策略合不合理  
5. prompt 是否要求模型誠實回答不知道

這些因素都會影響最後效果。

In [7]:
# 使用者查詢
query = "Agent 是什麼？"
query_vector = embedding_model.encode([f"query: {query}"], normalize_embeddings=True)

# 計算相似度
similarities = cosine_similarity(query_vector, corpus_vectors)

# 找出最相似的文本
top_k_indices = similarities[0].argsort()[-2:][::-1]
retrieved_texts = [corpus[idx] for idx in top_k_indices]
print("檢索到的文本:", retrieved_texts)

# 整合檢索文本並傳入 LLM API
d = " ".join(retrieved_texts)
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": f"根據以下上文回答使用者問題：{d}"},
        {"role": "user", "content": query}
    ]
)

print("生成的回答:", completion.choices[0].message.content)


檢索到的文本: ['RAG 是結合檢索和生成技術的模型。', 'RAG 可以從外部資料庫檢索訊息來增強生成模型的回應。']
生成的回答: RAG 模型中的 Agent 是指使用者与模型互動的接口或程式。用戶可以向 Agent 提出問題或请求，Agent 再将请求传递给 RAG 模型，然後 RAG 模型會根據用戶的请求生成回應。 Agent 可以是命令列介面、網頁應用程式或其他任何可以讓用戶与 RAG 模型互動的介面。


## 進階概念：加入門檻值（threshold）

前面的做法是固定取前 2 筆（Top-2），這很常見，但有一個問題：

> 就算資料不夠相關，系統還是會硬挑 2 筆給模型。

這可能導致模型根據錯誤上下文作答。

所以更穩健的做法之一，是加入門檻值（threshold）：
- 只有相似度高於某個值的文本才保留下來
- 如果沒有夠像的內容，就寧可不要給模型

---

## 這樣做的價值
加入 threshold 後，系統會更像在做「有條件的檢索」：

- 找得到足夠相關資料 → 才回答
- 找不到夠相關資料 → 就保守處理

這對企業知識問答尤其重要，因為錯答有時比不知道更危險。

## 加上門檻值（threshold）的版本

這一段示範另一種常見作法：  
不是固定取前 2 筆，而是只保留「相似度高於某個門檻」的文本。

這樣的好處是：
- 如果真的找不到足夠相關的內容，就不要硬塞上下文給模型
- 可以降低錯誤檢索造成的幻覺風險


## 範例一：故意輸入拼錯的問題

這裡的問題是：`Agnet 是什麼？`  
你可以把它看成使用者打錯字的情境。

### 這格的教學意義
我們要觀察兩件事：

1. embedding 模型是否仍能抓到接近語意  
2. threshold 是否能幫助我們避免把低品質結果塞給 LLM

---

## 你可以思考
如果 `retrieved_texts` 幾乎是空的，代表什麼？

通常代表：
- 知識庫裡沒有相關資料
- 或拼字錯誤導致語意比對失敗
- 或 threshold 設得太高

這時候，一個好的系統應該傾向誠實回答「不知道」，  
而不是硬湊一個看似合理的答案。

In [8]:
# 使用者查詢
query = "Agnet 是什麼？"
query_vector = embedding_model.encode([f"query: {query}"], normalize_embeddings=True)

# 計算相似度
similarities = cosine_similarity(query_vector, corpus_vectors).flatten()

# 找出最相似的文本（篩選相似度 > threshold 的文本）
threshold = 0.3
retrieved_texts = [corpus[idx] for idx, sim in enumerate(similarities) if sim > threshold]
print("檢索到的文本:", retrieved_texts)

# 整合檢索文本並傳入 LLM API
d = " ".join(retrieved_texts)
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": f"根據以下上文回答使用者問題，如果上下文沒有提及就回答不知道：{d}"},
        {"role": "user", "content": query}
    ]
)

print("生成的回答:", completion.choices[0].message.content)


檢索到的文本: ['如何實作 RAG？', 'RAG 是結合檢索和生成技術的模型。', 'RAG 可以從外部資料庫檢索訊息來增強生成模型的回應。', '嵌入和向量資料庫是 RAG 的基礎。']
生成的回答: 不知道


## 範例二：問一個知識庫中真的有的問題

這裡再次問：`RAG 是什麼？`  
但這次不是固定取 Top-k，而是加入了 threshold 機制。

### 你應該觀察什麼？
請比較這一格和前面 Top-k 版本的差異：

- 檢索結果有沒有比較乾淨？
- 留下來的文本是否更相關？
- 最後的回答是否更聚焦？

---

## 觀念總結
在真實應用中，Top-k 與 threshold 常常會一起搭配調整。  
例如：

- 先取 Top-5
- 再過濾掉相似度低於 0.35 的結果
- 若最後沒有任何文本留下，就要求模型回答不知道

這樣通常比單純固定取前幾筆更穩健。

In [9]:
# 使用者查詢
query = "RAG 是什麼？"
query_vector = embedding_model.encode([f"query: {query}"], normalize_embeddings=True)

# 計算相似度
similarities = cosine_similarity(query_vector, corpus_vectors).flatten()

# 找出最相似的文本（篩選相似度 > threshold 的文本）
threshold = 0.3
retrieved_texts = [corpus[idx] for idx, sim in enumerate(similarities) if sim > threshold]
print("檢索到的文本:", retrieved_texts)

# 整合檢索文本並傳入 LLM API
d = " ".join(retrieved_texts)
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": f"根據以下上文回答使用者問題，如果上下文沒有提及就回答不知道：{d}"},
        {"role": "user", "content": query}
    ]
)

print("生成的回答:", completion.choices[0].message.content)


檢索到的文本: ['如何實作 RAG？', 'RAG 是結合檢索和生成技術的模型。', 'RAG 可以從外部資料庫檢索訊息來增強生成模型的回應。', '嵌入和向量資料庫是 RAG 的基礎。']
生成的回答: RAG 是結合檢索和生成技術的模型。它可以從外部資料庫檢索訊息來增強生成模型的回應。



## 延伸練習建議
如果你想把這份 Notebook 再往上升級，可以試著挑戰以下方向：

1. 把 corpus 改成你自己的課程資料或公司文件  
2. 把文本切成 chunks，而不是一句一句  
3. 改成從 CSV、PDF 或網頁載入資料  
4. 加入「若查不到就回答不知道」的更完整 prompt  
5. 改成使用向量資料庫，例如 FAISS 或 pgvector  
6. 比較不同 embedding 模型的效果差異

---

## 給學生的一句提醒
真正的 RAG，不是把 API 串起來而已。  
你真正要學會的是：

> 如何讓模型「根據正確資料」回答，而不是只回答得像真的。

這才是 RAG 的核心價值。

---
# 作業

請使用 Python 完成一個可以做到：
當使用者輸入：

```python
query = "如何提高肌肉量？"
```
 
系統能找出語意最接近的文章標題。

In [11]:
query = "如何增加肌肉量？"

documents = [
    "如何透過重量訓練增加肌肉量",
    "減脂期間的飲食控制技巧",
    "Python 資料分析入門指南",
    "如何改善睡眠品質",
    "提升心肺耐力的運動方式",
    "大型語言模型的工作原理",
    "高蛋白飲食對健身的幫助",
    "時間管理的五個技巧"
]

In [22]:
# 將文件轉換為向量
document_vectors = embedding_model.encode(documents, normalize_embeddings=True)
query_vector = embedding_model.encode([f"query: {query}"], normalize_embeddings=True)

# 計算相似度
similarities = cosine_similarity(query_vector, document_vectors).flatten()

# 找出最相似的文本（篩選相似度 top 5 且大於 threshold 的文本）
threshold = 0.35
top_k_indices = similarities.argsort()[-5:][::-1]
retrieved_texts = [documents[idx] for idx in top_k_indices if similarities[idx] > threshold]
print("檢索到的文本:", retrieved_texts)

檢索到的文本: ['如何透過重量訓練增加肌肉量', '提升心肺耐力的運動方式', '高蛋白飲食對健身的幫助', '如何改善睡眠品質', '減脂期間的飲食控制技巧']


我嘗試幾個不同 threshold，發現效果不太一樣。
最後發現 threshold 設在 0.35 的時候，配合上 top-5 的做法，能找到比較相關的文章。

In [23]:
# 利用檢索到的文本生成回答
d = " ".join(retrieved_texts)
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": f"根據以下上文回答使用者問題，如果上下文沒有提及就回答不知道：{d}"},
        {"role": "user", "content": query}
    ]
)
print("生成的回答:", completion.choices[0].message.content)

生成的回答: 根據一般健身知識，想要增加肌肉量可以透過以下方式：

1. **重量訓練**：進行 resistance training 或 weight training 可以幫助增加肌肉量。選擇適合自己的訓練計畫，例如使用啞鈴、杠鈴或機器訓練，針對不同的肌肉群进行訓練。
2. **進食足夠蛋白質**：攝取足夠的蛋白質是增加肌肉量的關鍵。每天的蛋白質攝取量應該達到每公斤體重1.2-1.6克，來支援肌肉生長和修復。
3. **飲食控制**：保持均衡的飲食，包括复雜碳水化合物、健康脂肪和新鮮水果蔬菜。同時應該盡量避免過量食用高糖、高鹽分和高油脂的食物。
4. **足夠休息**：充足的休息和睡眠對於肌肉生長和恢復極為重要。每天至少應該睡7-8小時，讓肌肉有足夠的時間恢復。
5. **逐漸增加訓練強度**：隨著時間的推移，逐漸增加訓練的強度和重量，可以繼續挑戰肌肉，促進其生長。

同時，也需要記住，增加肌肉量需要時間和耐心，同時也需要一個合理的訓練計畫和均衡的飲食。若無法確定適合自己的方法，可以諮詢專業的健身教練或營養師。
